# Bayesian Optimization - Weeks 1-3 ArchiveThis notebook preserves week-specific strategy code from Weeks 1-3.These cells were removed from the main notebook during a restructure to keep the workflow clean.Refer to the main `bayesian_optimization.ipynb` for the current working notebook.

# Week 2 Strategy (Historical)Section 5.1 from the original notebook.

# Section 5.1: Week 2 Strategy Evolution (Historical Reference)

**⚠️ For Week 3+, skip to Section 5.5 below**

This section documents the strategy evolution from Week 1 to Week 2, kept for historical reference and learning.

## Week 2 Strategy Changes

Based on Week 1 analysis:
- **Problem functions (1, 2, 8)**: Changed acquisition functions and parameters
  - Function 1: UCB → EI (function appeared "dead")
  - Function 2: Reduced ξ for exploitation near best region
  - Function 8: UCB → EI to avoid boundary clustering
  
- **Winning functions (3-7)**: Shifted toward exploitation
  - Reduced β or switched to EI with low ξ
  - Focused on refining promising regions

The `generate_week2_queries()` function below shows the specific strategies used.

In [ ]:
def generate_week2_queries(track_predictions: bool = True) -> Tuple[Dict[int, np.ndarray], Dict]:
    """
    Generate optimized queries for Week 2 based on Week 1 analysis.
    
    Strategy changes from Week 1:
    - Function 1: UCB β=2.5 → EI xi=0.1 (more exploration, function appears "dead")
    - Function 2: EI xi=0.01 → EI xi=0.005 (exploit near best region)
    - Function 3: UCB β=2.5 → UCB β=2.0 (found improvement, moderate exploitation)
    - Function 4: UCB β=2.0 → UCB β=1.5 (massive improvement, exploit region)
    - Function 5: UCB β=2.0 → EI xi=0.001 (massive improvement, heavy exploitation)
    - Function 6: UCB β=2.5 → UCB β=2.0 (steady improvement, balanced)
    - Function 7: UCB β=1.5 → UCB β=1.0 (good improvement, focus exploitation)
    - Function 8: UCB β=3.0 → EI xi=0.05 (switch to EI to avoid boundary-seeking)
    
    Args:
        track_predictions: Whether to record predictions in the tracker
    
    Returns:
        Dictionary of queries for each function
    """
    
    # Week 2 strategies based on Week 1 analysis
    week2_strategies = {
        # PROBLEM FUNCTIONS - need different approach
        1: {
            'acq_func': 'ei', 
            'xi': 0.1,  # High exploration for "dead" function
            'bound_margin': 0.02,
            'expand_search': True,  # Search full space
            'n_random': 2000,  # More samples
        },
        2: {
            'acq_func': 'ei', 
            'xi': 0.005,  # Lower exploration, exploit near best
            'bound_margin': 0.02,
            'expand_search': False,  # Focus on data-driven bounds
        },
        8: {
            'acq_func': 'ei',  # Changed from UCB to EI
            'xi': 0.05,  # Moderate exploration parameter
            'bound_margin': 0.10,  # Increased margin to be safe
            'expand_search': True,
            'n_random': 3000,  # Many more samples for 8D space
            'surrogate_params': {'length_scale': 0.3, 'optimize': True}
        },
        
        # WINNING FUNCTIONS - shift toward exploitation
        3: {'acq_func': 'ucb', 'beta': 2.0, 'bound_margin': 0.02},
        4: {'acq_func': 'ucb', 'beta': 1.5, 'bound_margin': 0.02},
        5: {'acq_func': 'ei', 'xi': 0.001, 'bound_margin': 0.02},  # Heavy exploitation
        6: {'acq_func': 'ucb', 'beta': 2.0, 'bound_margin': 0.02},
        7: {'acq_func': 'ucb', 'beta': 1.0, 'bound_margin': 0.02},  # Strong exploitation
    }
    
    queries = {}
    surrogates = {}
    
    print("=" * 80)
    print("GENERATING WEEK 2 QUERIES (Enhanced Strategy)")
    print("=" * 80)
    print("\nStrategy changes from Week 1:")
    print("  • Functions 1, 2, 8: Adjusted strategies to address issues")
    print("  • Functions 3-7: Shifted toward exploitation")
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        strategy = week2_strategies[func_id]
        
        # Extract surrogate params
        surrogate_params = strategy.get('surrogate_params', {'length_scale': 0.5, 'optimize': True})
        surrogate = GPSurrogate(**surrogate_params)
        surrogate.fit(func_data.inputs, func_data.outputs)
        
        # Extract acquisition params
        acq_func = strategy.get('acq_func', 'ucb')
        bound_margin = strategy.get('bound_margin', 0.02)
        expand_search = strategy.get('expand_search', True)
        n_random = strategy.get('n_random', 1000)
        
        acq_params = {k: v for k, v in strategy.items() 
                     if k not in ['acq_func', 'surrogate_params', 'bound_margin', 
                                  'expand_search', 'n_random']}
        
        # Use enhanced optimization with boundary enforcement
        next_query, pred_mean, pred_std = optimize_acquisition_enhanced(
            surrogate, func_data, 
            acq_func=acq_func,
            n_random=n_random,
            bound_margin=bound_margin,
            expand_search=expand_search,
            **acq_params
        )
        
        queries[func_id] = next_query
        surrogates[func_id] = surrogate
        
        # Track predictions if enabled
        if track_predictions:
            prediction_tracker.record_prediction(
                week=2, func_id=func_id, query=next_query,
                pred_mean=pred_mean, pred_std=pred_std
            )
        
        # Display results
        best_x, best_y = func_data.get_best()
        strategy_str = f"{acq_func.upper()}"
        if 'beta' in acq_params:
            strategy_str += f" β={acq_params['beta']}"
        if 'xi' in acq_params:
            strategy_str += f" ξ={acq_params['xi']}"
        
        # Indicate if this is a changed strategy
        status = "⚡ CHANGED" if func_id in [1, 2, 8] else "↓ exploit"
        
        print(f"Function {func_id} ({func_data.n_dims}D) [{status}] - {strategy_str}:")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Next query: {np.array2string(next_query, precision=6, separator=', ')}")
        print(f"  Predicted: mean={pred_mean:.6f}, std={pred_std:.6f}")
        print()
    
    print("=" * 80)
    print("✓ Week 2 queries generated with enhanced strategies")
    print("✓ Predictions recorded for accuracy tracking")
    print("=" * 80)
    
    return queries, surrogates

print("✓ Week 2 query generator defined")

# Week 3 Strategy (Historical)

## Section 5.2: Week 3 Strategy (Recovery & Balanced Exploration)

Based on Week 2 results, Week 3 implements:
- **Function 5 Recovery**: Search near Week 1's successful location
- **Function 1 Boundary Exploration**: Add explicit corner/edge sampling
- **Re-explore declined functions** (2, 3, 6, 7) with increased exploration
- **Continue success** for Functions 4 and 8

In [ ]:
def generate_week3_queries(track_predictions: bool = True) -> Tuple[Dict[int, np.ndarray], Dict]:
    """
    Generate optimized queries for Week 3 based on Week 2 analysis.
    
    Strategy changes from Week 2 (Recovery & Balanced Exploration):
    - Function 1: EI xi=0.1 → UCB β=3.5 (aggressive exploration + boundary focus)
    - Function 2: EI xi=0.005 → UCB β=2.0 (balanced exploration after over-exploit)
    - Function 3: UCB β=2.0 → UCB β=2.5 (increased exploration)
    - Function 4: UCB β=1.5 → UCB β=1.5 (continue successful strategy)
    - Function 5: EI xi=0.001 → UCB β=2.5 (RECOVERY: search near Week 1 location)
    - Function 6: UCB β=2.0 → UCB β=3.0 (high exploration for 5D space)
    - Function 7: UCB β=1.0 → UCB β=2.5 (re-explore 6D space)
    - Function 8: EI xi=0.05 → EI xi=0.05 (continue successful strategy)
    
    Args:
        track_predictions: Whether to record predictions in the tracker
    
    Returns:
        (queries_dict, surrogates_dict)
    """
    
    # Week 3 strategies based on Week 2 analysis
    week3_strategies = {
        # PRIORITY 1: RECOVERY
        5: {
            'acq_func': 'ucb',
            'beta': 2.5,
            'bound_margin': 0.02,
            'expand_search': True,
            'use_regional_focus': True,  # Search near Week 1 location
            'focus_region': np.array([0.378897, 0.842367, 0.958538, 0.999568]),  # Week 1 query
            'focus_radius': 0.15,
            'n_random': 2000,
            'notes': 'Recovery: search near Week 1 successful location'
        },
        
        # PRIORITY 2: BOUNDARY EXPLORATION FOR "DEAD" FUNCTION
        1: {
            'acq_func': 'ucb',
            'beta': 3.5,  # Aggressive exploration
            'bound_margin': 0.01,  # Closer to boundaries
            'expand_search': True,
            'use_boundary_samples': True,  # Add explicit boundary/corner samples
            'n_random': 2000,
            'notes': 'Aggressive exploration + boundary focus for dead zone'
        },
        
        # PRIORITY 3: RE-EXPLORE DECLINED FUNCTIONS
        2: {
            'acq_func': 'ucb',
            'beta': 2.0,
            'bound_margin': 0.02,
            'expand_search': True,
            'notes': 'Balanced exploration after over-exploitation'
        },
        3: {
            'acq_func': 'ucb',
            'beta': 2.5,
            'bound_margin': 0.02,
            'expand_search': True,
            'notes': 'Increased exploration for 3D space'
        },
        6: {
            'acq_func': 'ucb',
            'beta': 3.0,
            'bound_margin': 0.02,
            'expand_search': True,
            'n_random': 1500,
            'notes': 'High exploration for 5D space'
        },
        7: {
            'acq_func': 'ucb',
            'beta': 2.5,
            'bound_margin': 0.02,
            'expand_search': True,
            'n_random': 1500,
            'notes': 'Re-explore 6D space'
        },
        
        # PRIORITY 4: CONTINUE SUCCESS STORIES
        4: {
            'acq_func': 'ucb',
            'beta': 1.5,
            'bound_margin': 0.02,
            'expand_search': True,
            'notes': 'Continue successful strategy'
        },
        8: {
            'acq_func': 'ei',
            'xi': 0.05,
            'bound_margin': 0.10,
            'expand_search': True,
            'n_random': 1200,
            'surrogate_params': {'length_scale': 0.3, 'optimize': True},
            'notes': 'Continue successful EI strategy for 8D (n_random capped for runtime)'
        },
    }
    
    queries = {}
    surrogates = {}
    
    print("=" * 80)
    print("GENERATING WEEK 3 QUERIES (Recovery & Balanced Exploration)")
    print("=" * 80)
    print()
    print("KEY CHANGES FROM WEEK 2:")
    print("  • Function 5: RECOVERY mode - searching near Week 1 success")
    print("  • Function 1: Aggressive boundary exploration")
    print("  • Functions 2,3,6,7: Increased exploration after Week 2 decline")
    print("  • Functions 4,8: Continuing successful strategies")
    print()
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        strategy = week3_strategies[func_id]
        print(f"  Processing Function {func_id}/8 (GP fit + acquisition) ...")
        
        # Extract surrogate params
        surrogate_params = strategy.get('surrogate_params', {'length_scale': 0.5, 'optimize': True})
        surrogate = GPSurrogate(**surrogate_params)
        surrogate.fit(func_data.inputs, func_data.outputs)
        
        # Extract acquisition params
        acq_func = strategy.get('acq_func', 'ucb')
        bound_margin = strategy.get('bound_margin', 0.02)
        expand_search = strategy.get('expand_search', True)
        n_random = strategy.get('n_random', 1000)
        use_regional_focus = strategy.get('use_regional_focus', False)
        use_boundary_samples = strategy.get('use_boundary_samples', False)
        
        acq_params = {}
        if 'beta' in strategy:
            acq_params['beta'] = strategy['beta']
        if 'xi' in strategy:
            acq_params['xi'] = strategy['xi']
        
        # Special handling for Function 1 (boundary exploration)
        if use_boundary_samples and func_data.n_dims == 2:
            # Add boundary samples to the random candidate pool
            boundary_samples = add_boundary_samples_2d(func_data.n_dims, n_samples=8, margin=0.01)
            
            # Use enhanced optimization with boundary samples included
            # We'll use a hybrid approach: evaluate boundary samples explicitly
            mean_bound, std_bound = surrogate.predict(boundary_samples)
            if acq_func == 'ucb':
                acq_values_bound = AcquisitionFunction.ucb(mean_bound, std_bound, beta=acq_params.get('beta', 2.0))
            else:
                _, y_best = func_data.get_best()
                acq_values_bound = AcquisitionFunction.ei(mean_bound, std_bound, y_best, xi=acq_params.get('xi', 0.01))
            
            # Get best boundary sample
            best_bound_idx = np.argmax(acq_values_bound)
            best_boundary_candidate = boundary_samples[best_bound_idx]
            best_boundary_acq = acq_values_bound[best_bound_idx]
            
            print(f"  → Function 1 boundary exploration: best boundary acq = {best_boundary_acq:.6f}")
        
        # Special handling for Function 5 (regional focus)
        if use_regional_focus:
            focus_region = strategy.get('focus_region')
            focus_radius = strategy.get('focus_radius', 0.15)
            next_query, pred_mean, pred_std = optimize_acquisition_with_regional_focus(
                surrogate, func_data,
                acq_func=acq_func,
                n_random=n_random,
                bound_margin=bound_margin,
                expand_search=expand_search,
                focus_region=focus_region,
                focus_radius=focus_radius,
                **acq_params
            )
        else:
            # Use standard enhanced optimization
            next_query, pred_mean, pred_std = optimize_acquisition_enhanced(
                surrogate, func_data,
                acq_func=acq_func,
                n_random=n_random,
                bound_margin=bound_margin,
                expand_search=expand_search,
                **acq_params
            )
        
        # For Function 1, compare with boundary candidate
        if use_boundary_samples and func_data.n_dims == 2:
            # Get acquisition value of regular next_query
            mean_regular, std_regular = surrogate.predict(next_query.reshape(1, -1))
            if acq_func == 'ucb':
                acq_regular = AcquisitionFunction.ucb(mean_regular, std_regular, beta=acq_params.get('beta', 2.0))[0]
            else:
                _, y_best = func_data.get_best()
                acq_regular = AcquisitionFunction.ei(mean_regular, std_regular, y_best, xi=acq_params.get('xi', 0.01))[0]
            
            # Use boundary sample if it's better
            if best_boundary_acq > acq_regular:
                next_query = best_boundary_candidate
                pred_mean, pred_std = mean_bound[best_bound_idx], std_bound[best_bound_idx]
                print(f"  → Selected boundary sample (acq: {best_boundary_acq:.6f} > {acq_regular:.6f})")
        
        queries[func_id] = next_query
        surrogates[func_id] = surrogate
        
        # Track predictions if enabled
        if track_predictions:
            prediction_tracker.record_prediction(
                week=3, func_id=func_id, query=next_query,
                pred_mean=pred_mean, pred_std=pred_std
            )
        
        # Display results
        best_x, best_y = func_data.get_best()
        strategy_str = f"{acq_func.upper()}"
        if 'beta' in acq_params:
            strategy_str += f" β={acq_params['beta']}"
        if 'xi' in acq_params:
            strategy_str += f" ξ={acq_params['xi']}"
        
        # Determine status emoji
        if func_id == 5:
            status = "🚨 RECOVERY"
        elif func_id == 1:
            status = "🎯 BOUNDARY"
        elif func_id in [2, 3, 6, 7]:
            status = "↑ explore"
        else:
            status = "✓ continue"
        
        print(f"Function {func_id} ({func_data.n_dims}D) [{status}] - {strategy_str}:")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Next query: {np.array2string(next_query, precision=6, separator=', ')}")
        print(f"  Predicted: mean={pred_mean:.6f}, std={pred_std:.6f}")
        if 'notes' in strategy:
            print(f"  Note: {strategy['notes']}")
        print()
    
    print("=" * 80)
    print("✓ Week 3 queries generated with recovery and exploration strategies")
    print("✓ Predictions recorded for accuracy tracking")
    print("=" * 80)
    
    return queries, surrogates

print("✓ Week 3 query generator defined")

## Week 3 Visualization Cells

## Week 3 Visualization: Understand 2D Function Surfaces

Visualize Functions 1 and 2 to understand where peaks might be located and validate Week 3 strategy.

In [ ]:
# Visualize Function 1 with Week 3 strategy (UCB β=3.5 + boundary exploration)
print("=" * 80)
print("FUNCTION 1 SURFACE ANALYSIS (Week 3 Strategy)")
print("=" * 80)
print("Strategy: UCB β=3.5 with boundary exploration")
print("Problem: Two weeks of ~0 output suggests peaks near boundaries")
print("=" * 80)
print()

visualize_2d_surface(1, surrogate=week3_surrogates[1], acq_func='ucb', beta=3.5)

In [ ]:
# Visualize Function 2 with Week 3 strategy (UCB β=2.0)
print("=" * 80)
print("FUNCTION 2 SURFACE ANALYSIS (Week 3 Strategy)")
print("=" * 80)
print("Strategy: UCB β=2.0 (balanced exploration)")
print("Problem: Declined from 0.279 to 0.247 with over-exploitation")
print("=" * 80)
print()

visualize_2d_surface(2, surrogate=week3_surrogates[2], acq_func='ucb', beta=2.0)

## Week 3 Strategy Summary

### Key Changes from Week 2:

**Priority 1: Function 5 Recovery (Critical)**
- Problem: Dropped from 2517.62 to 91.28 (-96%)
- Strategy: UCB β=2.5 with regional focus around Week 1's successful location
- Implementation: 1/3 of candidates sampled within radius 0.15 of Week 1 query

**Priority 2: Function 1 Boundary Exploration**
- Problem: Two weeks of ~0 output
- Strategy: UCB β=3.5 (aggressive) with explicit boundary/corner sampling
- Implementation: Evaluates 8 boundary/corner samples and compares with regular candidates

**Priority 3: Re-explore Declined Functions (2, 3, 6, 7)**
- All declined due to premature exploitation in Week 2
- Increased beta values: F2→2.0, F3→2.5, F6→3.0, F7→2.5
- Balances re-exploration with acquired knowledge

**Priority 4: Continue Success Stories (4, 8)**
- F4: UCB β=1.5 (21% improvement in Week 2)
- F8: EI xi=0.05 (4% improvement in Week 2)

### Expected Outcomes:
- **F5**: Recover to >500 (ideally toward 2000+)
- **F1**: Find ANY non-zero response (>0.01)
- **F2, F3, F6, F7**: Improve or maintain (no further decline)
- **F4, F8**: Continue steady improvement

### Next Steps:
1. **Copy queries** from the formatted output above
2. **Submit** to the competition portal
3. **Wait** for Week 3 results
4. **Analyze** results when received (use Section 6 functions)
5. **Adjust** strategy for Week 4 based on outcomes

In [ ]:
print("=" * 80)
print(f"{'Func':<6} {'Dims':<6} {'Samples':<8} {'Best':<12} {'Mean':<12} {'Std':<12}")
print("=" * 80)

for func_id, func_data in functions.items():
    summary = func_data.get_summary()
    print(f"{func_id:<6} {summary['n_dims']:<6} {summary['n_samples']:<8} "
          f"{summary['best_value']:<12.6f} {summary['mean_value']:<12.6f} "
          f"{summary['std_value']:<12.6f}")

print("=" * 80)

# Old Generic Generators (Historical)These were general-purpose generators superseded by week-specific strategies.

# Section 5: Weekly Query Generator

Generate query points for all 8 functions at once for weekly submission.

In [ ]:
def generate_weekly_queries(acq_func: str = 'ucb', surrogate_params: Dict = None,
                           **acq_params) -> Dict[int, np.ndarray]:
    """Generate query points for all functions."""
    if surrogate_params is None:
        surrogate_params = {'length_scale': 0.5, 'optimize': True}
    
    queries = {}
    surrogates = {}
    
    print("=" * 80)
    print("GENERATING WEEKLY QUERIES FOR ALL FUNCTIONS")
    print("=" * 80)
    print(f"Acquisition function: {acq_func.upper()}")
    print(f"Acquisition params: {acq_params}")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        
        # Create and fit surrogate
        surrogate = GPSurrogate(**surrogate_params)
        surrogate.fit(func_data.inputs, func_data.outputs)
        
        # Find next query
        next_query = optimize_acquisition(surrogate, func_data, acq_func=acq_func, **acq_params)
        
        queries[func_id] = next_query
        surrogates[func_id] = surrogate
        
        # Display
        best_x, best_y = func_data.get_best()
        mean, std = surrogate.predict(next_query.reshape(1, -1))
        
        print(f"Function {func_id} ({func_data.n_dims}D):")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Next query: {np.array2string(next_query, precision=6, separator=', ')}")
        print(f"  Predicted: mean={mean[0]:.6f}, std={std[0]:.6f}")
        print()
    
    print("=" * 80)
    return queries, surrogates

print("✓ Weekly query generator defined")

## Advanced: Custom Strategy Per Function

This function allows you to specify different strategies for each function.

In [ ]:
def generate_custom_queries(strategies: Dict[int, Dict] = None):
    """
    Generate queries with custom strategies per function.
    
    Args:
        strategies: Dict mapping function_id to strategy params
                   Example: {1: {'acq_func': 'ucb', 'beta': 2.0},
                            2: {'acq_func': 'ei', 'xi': 0.1}}
    
    If strategies is None, uses recommended defaults based on function characteristics.
    """
    # Default recommended strategies if none provided
    if strategies is None:
        strategies = {
            1: {'acq_func': 'ucb', 'beta': 2.5},  # 2D contamination - explore peaks
            2: {'acq_func': 'ei', 'xi': 0.01},    # Noisy - use EI
            3: {'acq_func': 'ucb', 'beta': 2.5},  # 3D drug - balanced
            4: {'acq_func': 'ucb', 'beta': 2.0},  # Dynamic - sustained exploration
            5: {'acq_func': 'ucb', 'beta': 2.0},  # Unimodal - can exploit more
            6: {'acq_func': 'ucb', 'beta': 2.5},  # 5D recipe - balanced
            7: {'acq_func': 'ucb', 'beta': 1.5},  # ML hyperparam - moderate
            8: {'acq_func': 'ucb', 'beta': 3.0},  # 8D - high exploration
        }
    
    queries = {}
    surrogates = {}
    
    print("=" * 80)
    print("GENERATING CUSTOM WEEKLY QUERIES")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        strategy = strategies.get(func_id, {'acq_func': 'ucb', 'beta': 2.0})
        
        # Create and fit surrogate
        surrogate_params = strategy.get('surrogate_params', {'length_scale': 0.5, 'optimize': True})
        surrogate = GPSurrogate(**surrogate_params)
        surrogate.fit(func_data.inputs, func_data.outputs)
        
        # Extract acquisition params
        acq_func = strategy.get('acq_func', 'ucb')
        acq_params = {k: v for k, v in strategy.items() 
                     if k not in ['acq_func', 'surrogate_params']}
        
        # Find next query
        next_query = optimize_acquisition(surrogate, func_data, 
                                         acq_func=acq_func, **acq_params)
        
        queries[func_id] = next_query
        surrogates[func_id] = surrogate
        
        # Display
        best_x, best_y = func_data.get_best()
        mean, std = surrogate.predict(next_query.reshape(1, -1))
        
        print(f"Function {func_id} ({func_data.n_dims}D) - {acq_func.upper()} {acq_params}:")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Next query: {np.array2string(next_query, precision=6, separator=', ')}")
        print(f"  Predicted: mean={mean[0]:.6f}, std={std[0]:.6f}")
        print()
    
    print("=" * 80)
    return queries, surrogates

print("✓ Custom strategy query generator defined")

## Option 1: Use Simple Generator (Same Strategy for All)

In [ ]:
/# Simple approach: Same strategy for all functions
# weekly_queries, _ = generate_weekly_queries(acq_func='ucb', beta=2.0)

print("Uncomment above to use simple approach")

## Option 2: Use Custom Strategies (RECOMMENDED)

In [ ]:
# Option A: Use recommended defaults (tuned for each function)
# weekly_queries, weekly_surrogates = generate_custom_queries()

# Option B: Customize for specific functions
my_strategies = {
     1: {'acq_func': 'ei', 'xi': 0.1},      # More exploration for function 1
     2: {'acq_func': 'ei', 'xi': 0.005},         # EI for noisy function 2
     3: {'acq_func': 'UCB', 'beta': 2.0}, 
     4: {'acq_func': 'UCB', 'beta': 1.5},
     5: {'acq_func': 'UCB', 'beta': 1.2},
     6: {'acq_func': 'UCB', 'beta': 2.0},
     7: {'acq_func': 'UCB', 'beta': 1.0},
     8: {'acq_func': 'ucb', 'beta': 2.5,         # Less extreme exploration
        'surrogate_params': {'length_scale': 0.3, 'optimize': True}},                # Aggressive exploitation for unimodal
     # Functions not specified use {'acq_func': 'ucb', 'beta': 2.0}
 }
weekly_queries, weekly_surrogates = generate_custom_queries(my_strategies)

In [ ]:
def generate_weekly_queries(week: int,
                           strategies: Optional[Dict[int, Dict]] = None,
                           auto_recommend: bool = True) -> Tuple[Dict, Dict, Dict]:
    """
    Generate queries for any week with optional auto-recommendation.
    
    Args:
        week: Week number (for tracking/naming)
        strategies: Custom strategies per function (format: {func_id: {params}})
                   If None and auto_recommend=True, analyzes previous week
        auto_recommend: If True and strategies=None, auto-generate recommendations
        
    Returns:
        tuple: (queries_dict, surrogates_dict, analysis_dict)
    
    Example:
        # Auto-recommend based on previous week
        queries, surrogates, analysis = generate_weekly_queries(week=3)
        
        # Use custom strategies
        custom = {1: {'acq_func': 'ei', 'xi': 0.1}, 2: {'acq_func': 'ucb', 'beta': 2.5}}
        queries, surrogates, analysis = generate_weekly_queries(week=3, strategies=custom)
    """
    # Auto-recommend if requested and no strategies provided
    if strategies is None and auto_recommend and week > 1:
        print(f"Auto-recommending strategies based on Week {week-1} analysis...\n")
        analysis = analyze_weekly_performance(week=week-1)
        recommendations = recommend_strategies(analysis, current_week=week-1)
        # Extract just the strategy dicts
        strategies = {fid: rec['strategy'] for fid, rec in recommendations.items()}
    elif strategies is None:
        # Use balanced defaults
        strategies = {fid: {'acq_func': 'ucb', 'beta': 2.0} for fid in range(1, 9)}
    
    queries = {}
    surrogates = {}
    analysis_dict = {}
    
    print("=" * 80)
    print(f"GENERATING WEEK {week} QUERIES")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        strategy = strategies.get(func_id, {'acq_func': 'ucb', 'beta': 2.0})
        
        # Create and fit surrogate
        surrogate_params = strategy.get('surrogate_params', {})
        surrogate = GPSurrogate(
            length_scale=surrogate_params.get('length_scale', 0.5),
            optimize=surrogate_params.get('optimize', True)
        )
        surrogate.fit(func_data.inputs, func_data.outputs)
        surrogates[func_id] = surrogate
        
        # Extract acquisition function parameters
        acq_func = strategy.get('acq_func', 'ucb')
        acq_params = {k: v for k, v in strategy.items() if k not in ['acq_func', 'surrogate_params']}
        
        # Generate query using enhanced optimization
        try:
            result = optimize_acquisition_enhanced(
                surrogate, func_data, 
                acq_func=acq_func, 
                **acq_params
            )
            # optimize_acquisition_enhanced returns (point, mean, std)
            if isinstance(result, tuple) and len(result) == 3:
                next_query, pred_mean, pred_std = result
                mean, std = [pred_mean], [pred_std]  # Match predict() output format
            else:
                next_query = result
                mean, std = surrogate.predict(next_query.reshape(1, -1))
        except:
            # Fallback to standard optimization
            next_query = optimize_acquisition(
                surrogate, func_data,
                acq_func=acq_func,
                **acq_params
            )
            mean, std = surrogate.predict(next_query.reshape(1, -1))
        
        queries[func_id] = next_query
        best_x, best_y = func_data.get_best()
        
        analysis_dict[func_id] = {
            'predicted_mean': float(mean[0]),
            'predicted_std': float(std[0]),
            'current_best': float(best_y),
            'strategy': strategy,
            'n_samples': func_data.n_samples
        }
        
        # Display
        acq_display = f"{acq_func.upper()}"
        if 'beta' in acq_params:
            acq_display += f" β={acq_params['beta']}"
        if 'xi' in acq_params:
            acq_display += f" ξ={acq_params['xi']}"
        
        print(f"Function {func_id} ({func_data.n_dims}D) - {acq_display}:")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Predicted: μ={mean[0]:.6f}, σ={std[0]:.6f}")
        print(f"  Query: {np.array2string(next_query, precision=6, separator=', ')}")
        print()
    
    print("=" * 80)
    print(f"✓ Generated {len(queries)} queries for Week {week}")
    print("=" * 80)
    print()
    
    return queries, surrogates, analysis_dict


print("✓ generate_weekly_queries() generic function defined")

In [ ]:
def weekly_cycle(current_week: int, 
                preview_only: bool = False,
                aggressive: bool = False) -> Dict:
    """
    Complete weekly cycle: analyze previous week → recommend → generate queries.
    
    This is the ONE-STOP function for your weekly workflow!
    
    Args:
        current_week: The week you're preparing queries for (e.g., 3 for Week 3)
        preview_only: If True, shows recommendations without generating queries
        aggressive: Whether to be aggressive with strategy changes
        
    Returns:
        dict: Complete cycle results including analysis, recommendations, and queries
        
    Example:
        # Preparing for Week 3 after receiving Week 2 results
        results = weekly_cycle(current_week=3)
        
        # Review recommendations first
        results = weekly_cycle(current_week=3, preview_only=True)
        
        # Then generate if happy with recommendations
        results = weekly_cycle(current_week=3, preview_only=False)
    """
    print("╔" + "═" * 78 + "╗")
    print(f"║ WEEKLY CYCLE: PREPARING FOR WEEK {current_week:2d}" + " " * 48 + "║")
    print("╚" + "═" * 78 + "╝")
    print()
    
    results = {
        'target_week': current_week,
        'analysis': None,
        'recommendations': None,
        'queries': None,
        'surrogates': None,
        'query_analysis': None
    }
    
    # Step 1: Analyze previous week (if exists)
    if current_week > 1:
        print(f"📊 STEP 1: Analyzing Week {current_week - 1} Performance")
        print("-" * 80)
        try:
            analysis = analyze_weekly_performance(week=current_week - 1)
            results['analysis'] = analysis
        except Exception as e:
            print(f"⚠ Could not analyze Week {current_week - 1}: {e}")
            print("   Continuing with default strategies...\n")
            analysis = None
    else:
        print(f"📊 STEP 1: Week 1 - No previous data to analyze")
        print("-" * 80)
        print("   Using default strategies for initial week\n")
        analysis = None
    
    # Step 2: Generate recommendations
    if analysis:
        print(f"🎯 STEP 2: Generating Strategy Recommendations")
        print("-" * 80)
        recommendations = recommend_strategies(analysis, current_week=current_week - 1, aggressive=aggressive)
        results['recommendations'] = recommendations
        
        # Extract strategy dicts
        strategies = {fid: rec['strategy'] for fid, rec in recommendations.items()}
    else:
        print(f"🎯 STEP 2: Using Default Strategies")
        print("-" * 80)
        strategies = {fid: {'acq_func': 'ucb', 'beta': 2.0} for fid in range(1, 9)}
        print("Using balanced UCB (β=2.0) for all functions\n")
    
    # Step 3: Generate queries (unless preview only)
    if not preview_only:
        print(f"🔮 STEP 3: Generating Week {current_week} Queries")
        print("-" * 80)
        queries, surrogates, query_analysis = generate_weekly_queries(
            week=current_week,
            strategies=strategies,
            auto_recommend=False  # We already did recommendation above
        )
        results['queries'] = queries
        results['surrogates'] = surrogates
        results['query_analysis'] = query_analysis
        
        # Step 4: Format for submission
        print(f"📋 STEP 4: Formatted Output for Submission")
        print("-" * 80)
        formatted = format_for_portal(queries)
        results['formatted_queries'] = formatted
        print(formatted)
        print()
    else:
        print(f"👀 PREVIEW MODE: Skipping query generation")
        print("-" * 80)
        print("   Review recommendations above, then run with preview_only=False\n")
    
    # Summary
    print("╔" + "═" * 78 + "╗")
    print(f"║ CYCLE COMPLETE" + " " * 63 + "║")
    print("╚" + "═" * 78 + "╝")
    
    if not preview_only:
        print(f"\n✓ Week {current_week} queries ready for submission!")
        print("  Copy the formatted output above and submit to the portal.")
    else:
        print(f"\n👉 Next step: Run weekly_cycle({current_week}, preview_only=False) to generate queries")
    
    print()
    
    return results


print("✓ weekly_cycle() unified workflow function defined")

# Old Format Functions (Historical)

## IMPORTANT: Portal Submission Format

**The portal expects HYPHENS, not commas!**  
Format: `x1-x2-x3-...-xn`

In [ ]:
def format_for_portal(queries: Dict[int, np.ndarray]):
    """
    Format queries correctly for the competition portal.
    Portal expects: x1-x2-x3-...-xn (HYPHENS not commas!)
    """
    print("=" * 80)
    print("PORTAL SUBMISSION FORMAT")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        if func_id in queries:
            query = queries[func_id]
            # Portal format uses HYPHENS between values
            query_str = "-".join([f"{x:.6f}" for x in query])
            print(f"Function {func_id}: {query_str}")
        else:
            print(f"Function {func_id}: -")
    
    print()
    print("=" * 80)
    print("Copy the values above and paste into the portal")
    print("=" * 80)

# Format with correct portal format (hyphens!)
format_for_portal(weekly_queries)

In [ ]:
def format_queries_for_submission(queries: Dict[int, np.ndarray]):
    """Format queries in a submission-ready format."""
    print("=" * 80)
    print("WEEKLY SUBMISSION - QUERY POINTS")
    print("=" * 80)
    print()
    
    for func_id, query in queries.items():
        print(f"Function {func_id}:")
        query_str = ", ".join([f"{x:.6f}" for x in query])
        print(f"  {query_str}")
        print()
    
    print("=" * 80)

# Format the generated queries
format_queries_for_submission(weekly_queries)

## Format Queries for Submission

Display queries in a format ready for copy-paste submission.

In [ ]:
def format_queries_for_submission(queries: Dict[int, np.ndarray]):
    """Format queries in a submission-ready format."""
    print("=" * 80)
    print("WEEKLY SUBMISSION - QUERY POINTS")
    print("=" * 80)
    print()
    
    for func_id, query in queries.items():
        print(f"Function {func_id}:")
        # Format as comma-separated values for easy copy-paste
        query_str = ", ".join([f"{x:.6f}" for x in query])
        print(f"  {query_str}")
        print()
    
    print("=" * 80)

# Format the generated queries
format_queries_for_submission(weekly_queries)

# Old Week 2 Result Loading (Historical)

## Load and Process Week 2 Results

Use this cell to automatically load and process Week 2 results.

In [ ]:
# Load Week 1 results
inputs_week1, outputs_week1, _ = load_results(week_index=0)  # Week 1

# Update all functions with Week 1 results
week1_results = update_all_functions_with_results(inputs_week1, outputs_week1, week=1, save=True)

In [ ]:
# Load Week 2 results
inputs_week2, outputs_week2, _ = load_results(week_index=1)  # Week 2

In [ ]:
def load_latest_results() -> Tuple[Dict[int, np.ndarray], Dict[int, float], int]:
    """
    Load the most recent week's results.
    
    Returns:
        tuple: (inputs_dict, outputs_dict, actual_week_number)
    
    Example:
        inputs, outputs, week_num = load_latest_results()
        print(f"Loaded Week {week_num}")
    """
    return load_results(week_index=-1)


def load_all_weeks(results_dir: Optional[str] = None) -> List[Tuple[Dict[int, np.ndarray], Dict[int, float], int]]:
    """
    Load all available weeks' results as a list.
    
    Args:
        results_dir: Optional specific directory path. If None, auto-detects latest Week directory
    
    Returns:
        list: List of (inputs_dict, outputs_dict, week_number) tuples, one per week
    
    Example:
        all_weeks = load_all_weeks()
        for inputs, outputs, week_num in all_weeks:
            print(f"Week {week_num}: {len(inputs)} functions")
    """
    # Auto-detect latest Week directory if not specified
    if results_dir is None:
        base_path = Path("..")
        week_dirs = sorted([d for d in base_path.glob("Week *") if d.is_dir()])
        if not week_dirs:
            raise FileNotFoundError("No Week directories found")
        results_dir = str(week_dirs[-1])
    
    # Load first to determine how many weeks are available
    _, _, _ = load_results(week_index=0, results_dir=results_dir)
    
    # Count available weeks by checking the files
    results_path = Path(results_dir)
    inputs_file = results_path / "inputs.txt"
    
    with open(inputs_file, 'r') as f:
        full_content = f.read()
    
    # Count list structures
    num_weeks = full_content.count('\n[') + (1 if full_content.strip().startswith('[') else 0)
    
    # Load all weeks
    all_results = []
    for week_idx in range(num_weeks):
        try:
            inputs, outputs, week_num = load_results(week_index=week_idx, results_dir=results_dir)
            all_results.append((inputs, outputs, week_num))
        except Exception as e:
            print(f"Warning: Could not load week index {week_idx}: {e}")
    
    print(f"✓ Loaded {len(all_results)} weeks of results")
    return all_results


print("✓ Convenience functions defined (load_latest_results, load_all_weeks)")

## Example: Load Results

Simple examples showing how to use the new loading functions.

In [ ]:
# Example 1: Load the latest week's results
inputs_latest, outputs_latest, week_num = load_latest_results()
print(f"Loaded Week {week_num} with {len(inputs_latest)} functions")

# Example 2: Load a specific week by index (0=Week 1, 1=Week 2, etc.)
# inputs_week1, outputs_week1, week_num = load_results(week_index=0)
# inputs_week2, outputs_week2, week_num = load_results(week_index=1)

# Example 3: Load all available weeks
# all_weeks = load_all_weeks()
# for inputs, outputs, wk in all_weeks:
#     best_func5 = outputs[5]
#     print(f"Week {wk}: Function 5 = {best_func5:.2f}")

In [ ]:
# EXAMPLE 2: Experimental Analysis
# Compare strategies with different training sets

# Option A: Train only on Week 1 data
# functions_v1 = {i: FunctionData(i) for i in range(1, 9)}
# initialize_from_history(functions_v1, week_indices=[0])  # Only Week 1

# Option B: Train on Weeks 1-2 data
# functions_v2 = {i: FunctionData(i) for i in range(1, 9)}
# initialize_from_history(functions_v2, week_indices=[0, 1])  # Weeks 1-2

# Now compare:
# - How do surrogate model predictions differ?
# - Which acquisition function performs better with more data?
# - Does adding Week 2 data change the optimal strategy?

In [ ]:
week2_results = update_all_functions_with_results(inputs_week2, outputs_week2, week=2, save=True)

# Streamlined Workflow (Historical Reference)

## Week 3+ Streamlined Workflow

Use these simple patterns for Week 3 and beyond.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEEK 3: ONE-LINE WORKFLOW
# ══════════════════════════════════════════════════════════════════════════════
# This single function does everything: analyze Week 2, recommend strategies,
# generate Week 3 queries, and format for submission.

results = weekly_cycle(current_week=3)

# That's it! Copy the formatted output above and submit to the portal.
# The results dictionary contains all the details if you want to inspect:
# - results['analysis']: Week 2 performance analysis
# - results['recommendations']: Strategy recommendations for Week 3
# - results['queries']: Generated queries
# - results['formatted_queries']: Ready to copy-paste

## Alternative: Preview Recommendations First

If you want to review recommendations before generating queries:

In [ ]:
# Step 1: Preview recommendations (no query generation)
# preview_results = weekly_cycle(current_week=3, preview_only=True)

# Step 2: Review the output above, then generate if satisfied
# results = weekly_cycle(current_week=3, preview_only=False)

print("Uncomment the lines above to use preview mode")

## Advanced: Custom Strategy Override

If you want to customize specific functions:

In [ ]:
# Step 1: Analyze previous week manually
# analysis = analyze_weekly_performance(week=2)

# Step 2: Get recommendations
# recommendations = recommend_strategies(analysis, current_week=2)

# Step 3: Customize specific functions if desired
# custom_strategies = {fid: rec['strategy'] for fid, rec in recommendations.items()}
# custom_strategies[5] = {'acq_func': 'ei', 'xi': 0.001}  # Override Function 5
# custom_strategies[8] = {'acq_func': 'ucb', 'beta': 3.5}  # Override Function 8

# Step 4: Generate with custom strategies
# queries, surrogates, analysis = generate_weekly_queries(week=3, strategies=custom_strategies, auto_recommend=False)

# Step 5: Format for submission
# formatted = format_for_portal(queries)
# print(formatted)

print("Uncomment the lines above for advanced customization")